###Ejercicio 1):
 Utilice lo aprendido en los trabajos prácticos previos (uso de biopython para obtenersecuencias, BLAST y el “parser del mismo”) para:

 i) obtener 10 secuencias pertenecientes a una misma familia y

 ii) Alinearlas y obtener la correspondiente matriz de puntaje / identidad

 Analice si la matriz es (o no) simetrica

 Compare matrices utilizando diferentes puntajes que le otorga BLAST (Bit Score, Identidad, E-Value)

Lista de 10 proteinas CYP1A1 de distintas especies

In [1]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 27.6 MB/s eta 0:00:00


In [2]:
from Bio import SeqIO, pairwise2
from Bio.Align import substitution_matrices
import numpy as np
import pandas as pd

from Bio import Phylo
from io import StringIO


/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [3]:
!wget ftp://ftp.ncbi.nlm.nih.gov/blast/executables/blast+/LATEST/ncbi-blast-*-x64-linux.tar.gz
!tar -xzf ncbi-blast-*-x64-linux.tar.gz
!mv ncbi-blast-*/bin/* /usr/local/bin/


--2025-09-25 15:06:10--  ftp://ftp.ncbi.nlm.nih.gov/blast/executables/blast+/LATEST/ncbi-blast-*-x64-linux.tar.gz
           => ‘.listing’
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.7, 130.14.250.11, 130.14.250.12, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.7|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /blast/executables/blast+/LATEST ... done.
==> PASV ... done.    ==> LIST ... done.

.listing                [ <=>                ]   2.62K  --.-KB/s    in 0.01s   

2025-09-25 15:06:11 (212 KB/s) - ‘.listing’ saved [2678]

Removed ‘.listing’.
--2025-09-25 15:06:11--  ftp://ftp.ncbi.nlm.nih.gov/blast/executables/blast%2B/LATEST/ncbi-blast-2.17.0%2B-x64-linux.tar.gz
           => ‘ncbi-blast-2.17.0+-x64-linux.tar.gz’
==> CWD not required.
==> PASV ... done.    ==> RETR ncbi-blast-2.17.0+-x64-linux.tar.gz ... done.
Length: 296006458 (282M)

ncbi-blas

In [4]:
import pandas as pd
def cargar_blosum60(path="/content/blosum60.txt"):
    with open(path) as f:
        lines = [line.strip() for line in f if not line.startswith("#")]

    header = lines[0].split()
    data = []
    index = []

    for line in lines[1:]:
        parts = line.split()
        index.append(parts[0])
        data.append([int(x) for x in parts[1:]])

    df = pd.DataFrame(data, index=index, columns=header)
    return df

# Cargar matriz
blosum60_df = cargar_blosum60()

# Mostrar parte de la matriz

def df_to_blosum_dict(df):
    matrix = {}
    for i in df.index:
        for j in df.columns:
            score = df.at[i, j]
            matrix[(i, j)] = score
            matrix[(j, i)] = score  # aseguramos simetría
    return matrix


blosum_dict = df_to_blosum_dict(blosum60_df)
print(blosum_dict)


{('A', 'A'): np.int64(5), ('A', 'R'): np.int64(-2), ('R', 'A'): np.int64(-2), ('A', 'N'): np.int64(-1), ('N', 'A'): np.int64(-1), ('A', 'D'): np.int64(-2), ('D', 'A'): np.int64(-2), ('A', 'C'): np.int64(-1), ('C', 'A'): np.int64(-1), ('A', 'Q'): np.int64(-1), ('Q', 'A'): np.int64(-1), ('A', 'E'): np.int64(-1), ('E', 'A'): np.int64(-1), ('A', 'G'): np.int64(0), ('G', 'A'): np.int64(0), ('A', 'H'): np.int64(-2), ('H', 'A'): np.int64(-2), ('A', 'I'): np.int64(-2), ('I', 'A'): np.int64(-2), ('A', 'L'): np.int64(-2), ('L', 'A'): np.int64(-2), ('A', 'K'): np.int64(-1), ('K', 'A'): np.int64(-1), ('A', 'M'): np.int64(-1), ('M', 'A'): np.int64(-1), ('A', 'F'): np.int64(-3), ('F', 'A'): np.int64(-3), ('A', 'P'): np.int64(-1), ('P', 'A'): np.int64(-1), ('A', 'S'): np.int64(1), ('S', 'A'): np.int64(1), ('A', 'T'): np.int64(0), ('T', 'A'): np.int64(0), ('A', 'W'): np.int64(-4), ('W', 'A'): np.int64(-4), ('A', 'Y'): np.int64(-2), ('Y', 'A'): np.int64(-2), ('A', 'V'): np.int64(0), ('V', 'A'): np.int6

In [5]:
import os

def blast_two_seqs(seq_query, seq_db, id_query="query", id_db="db", temp_dir="temp_blast"):
    """
    Hace BLASTp local entre dos secuencias proteicas y devuelve métricas clave.

    Args:
      seq_query (str): Secuencia proteica que será la consulta.
      seq_db (str): Secuencia proteica que será la base de datos.
      id_query (str): ID para la secuencia consulta.
      id_db (str): ID para la secuencia base de datos.
      temp_dir (str): Carpeta temporal para archivos.

    Returns:
      tuple: (evalue, bitscore, pident) del mejor hit.
             Si no hay hits, devuelve (None, None, None).
    """
    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir)

    # Crear archivo fasta para consulta
    query_path = os.path.join(temp_dir, "query.fasta")
    with open(query_path, "w") as f:
        f.write(f">{id_query}\n{seq_query}\n")

    # Crear archivo fasta para base de datos
    db_path = os.path.join(temp_dir, "db.fasta")
    with open(db_path, "w") as f:
        f.write(f">{id_db}\n{seq_db}\n")

    # Crear base de datos local
    db_name = os.path.join(temp_dir, "db")
    cmd_make_db = f"makeblastdb -in {db_path} -dbtype prot -out {db_name} -logfile /dev/null"
    os.system(cmd_make_db)

    # Ejecutar blastp
    output_path = os.path.join(temp_dir, "blast_out.txt")
    cmd_blast = (
        f"blastp -query {query_path} -db {db_name} "
        f"-out {output_path} -outfmt '6 evalue bitscore pident' "
        f"-max_target_seqs 1 -num_threads 1"
    )
    os.system(cmd_blast)

    # Leer resultado
    evalue, bitscore, pident = None, None, None
    if os.path.exists(output_path):
        with open(output_path) as f:
            line = f.readline().strip()
            if line:
                evalue_str, bitscore_str, pident_str = line.split("\t")
                evalue = float(evalue_str)
                bitscore = float(bitscore_str)
                pident = float(pident_str)

    return evalue, bitscore, pident


#Obtenemos 10 secuencias de la familia quinasa

In [6]:

from Bio import Entrez, SeqIO

# Obligatorio para usar Entrez
Entrez.email = "joaquinsalva03@gmail.com"

# Paso 1: buscar proteínas asociadas a una familia en NCBI Protein
# Ejemplo: globinas
handle = Entrez.esearch(
    db="protein",
    term="Protein kinase[Protein Name]",
    retmax=10  # cantidad de secuencias a traer
)
record = Entrez.read(handle)
handle.close()

print("IDs encontrados:", record["IdList"])

# Paso 2: bajar las secuencias en formato fasta
handle = Entrez.efetch(
    db="protein",
    id=",".join(record["IdList"]),
    rettype="fasta",
    retmode="text"
)

# Paso 3: parsear a strings y guardarlas en una lista
lista_quinasas = [str(rec.seq) for rec in SeqIO.parse(handle, "fasta")]
handle.close()

print("Número de secuencias:", len(lista_quinasas))



IDs encontrados: ['3063888818', '3063884927', '3063834559', '3063804125', '2924247891', '2575255726', '2573173597', '2511548032', '2254702454', '1300052709']
Número de secuencias: 10


In [7]:
def guardar_lista_fasta(lista_secuencias, output_path="secuencias.fasta", prefix="Prot_"):
    """
    Guarda una lista de secuencias en formato FASTA.

    Args:
        lista_secuencias (list): Lista de secuencias (strings).
        output_path (str): Nombre del archivo de salida.
        prefix (str): Prefijo para los IDs de cada secuencia.
    """
    with open(output_path, "w") as f:
        for i, seq in enumerate(lista_secuencias):
            f.write(f">{prefix}{i}\n")
            f.write(f"{seq}\n")
guardar_lista_fasta(lista_quinasas, "quinasas.fasta")


In [8]:
# Cargar matriz
blosum60_df = cargar_blosum60()
def obtener_matriz_blast_scores(lista_proteinas, blosum):
    # Inicializar matriz
    n = len(lista_proteinas)
    evalue_matrix, bitscore_matrix, pident_matrix = np.zeros((n, n)), np.zeros((n, n)),np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            seq1 = lista_proteinas[i]
            seq2 = lista_proteinas[j]
            evalue, bitscore, pident = blast_two_seqs(seq1, seq2)
            evalue_matrix[i][j]= evalue
            bitscore_matrix[i][j] = bitscore
            pident_matrix[i][j]  =  pident


    maximo = np.nanmax(evalue_matrix)
    evalue_matrix = np.nan_to_num(evalue_matrix, nan=maximo)

    bitscore_matrix = np.nan_to_num(bitscore_matrix, nan=0)

    pident_matrix = np.nan_to_num(pident_matrix, nan=0)
    return evalue_matrix, bitscore_matrix, pident_matrix




Ejercicio 2): Programe un algoritmo que dada una matriz de distancias construya un árbol guía.Utilice el mismo para trabajar con las diferentes matrices obtenidas a partir de los puntajes deBLAST y analice los resultados de manera comparativa.

# De la matriz al arbol guia

## De la matriz al arbol

Tomamos la matriz de distancias y, reduciendo su dimension, calculamos el agrupamiento de nodos del arbol guia

Para esto tenemos que:

* Tener una matriz de distancias donde las columnas y las filas son un conjunto de secuencias (que lo llamamos C).
* encontrar el maximo valor de la matriz
  junto con sus coordenadas
  maximo = distancia (A,B), con A,B en C, y su distancia el maximo score, es decir, mayor parecido entre si.  
* Quito las filas y columnas donde se vea distancias de A o B a otra secuencia X
* Armo una fila que sea distancia ((AB, X)

De esta manera, evitamos recalcular toda la matriz de distancia, donde lo que nos interesa modificar es aquellas filas, columnas que se vean afectadas

## Graficar el arbol guia

In [9]:
import numpy as np

def imprimir_matriz(matriz, etiquetas):
    print("\nMatriz actual:")
    #encabezado = "\t" + "\t".join(etiquetas)
    #print(encabezado)
    #for etiqueta, fila in zip(etiquetas, matriz):
     #   print(etiqueta + "\t" + "\t".join(f"{val:.2f}" for val in fila))
    df = pd.DataFrame(matriz, index=etiquetas, columns=etiquetas)
    display(df)
    return
def encontrar_maximo(matriz):
    n = len(matriz)
    max_val = float('-inf')  # empezamos desde -infinito
    coords = (-1, -1)
    for i in range(n):
        for j in range(i+1, n):  # triángulo superior
            if matriz[i][j] > max_val:
                max_val = matriz[i][j]
                coords = (i, j)
    return coords, max_val

def encontrar_minimo(matriz):
    n = len(matriz)
    min_val = float('inf')  # empezamos desde -infinito
    coords = (-1, -1)
    for i in range(n):
        for j in range(i+1, n):  # triángulo superior
            if matriz[i][j] < min_val:
                min_val = matriz[i][j]
                coords = (i, j)
    return coords, min_val


def calcular_distancia_promedio(matriz, i, j):
    n = len(matriz)
    nueva_fila = []
    for k in range(n):
        if k != i and k != j:
            d = (matriz[i][k] + matriz[j][k]) / 2
            nueva_fila.append(d)
    return nueva_fila

def actualizar_matriz(matriz, etiquetas, i, j):
    # Unimos etiquetas como una string (sin espacios si lo querés tipo Newick)
    nueva_etiqueta = f"({etiquetas[i]},{etiquetas[j]})"

    nueva_fila = calcular_distancia_promedio(matriz, i, j)

    # Eliminar i y j (mayor índice primero)
    indices = sorted([i, j], reverse=True)
    for idx in indices:
        matriz = np.delete(matriz, idx, axis=0)
        matriz = np.delete(matriz, idx, axis=1)
        etiquetas.pop(idx)

    # Agregar nueva fila/columna
    matriz = np.vstack([matriz, nueva_fila])
    nueva_col = nueva_fila + [0.0]
    matriz = np.column_stack([matriz, nueva_col])
    etiquetas.append(nueva_etiqueta)

    return matriz, etiquetas



In [10]:
from Bio import SeqIO
from Bio.Align import substitution_matrices
from Bio.Align import PairwiseAligner
import numpy as np

# Cargar las secuencias
seqs = list(SeqIO.parse("quinasas.fasta", "fasta"))

# Configurar alineador
aligner = PairwiseAligner()
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

n = len(seqs)
dist_matrix = np.zeros((n, n))

# Calcular matriz de distancias (1 - identidad normalizada)
for i in range(n):
    for j in range(n):
        score = aligner.score(seqs[i].seq, seqs[j].seq)
        max_len = max(len(seqs[i].seq), len(seqs[j].seq))
        identity = score / max_len
        dist_matrix[i, j] = score

print("Matriz de distancias inicial:")
print(dist_matrix)



Matriz de distancias inicial:
[[ 2.3400e+03  1.2800e+02 -1.4000e+01  0.0000e+00 -2.1000e+01 -3.6500e+01
  -3.8500e+01 -9.3500e+01 -9.2500e+01 -4.5500e+01]
 [ 1.2800e+02  3.4680e+03 -5.0000e+01 -8.3500e+01 -4.1500e+01 -1.1050e+02
  -1.1750e+02 -9.1000e+01 -9.0000e+01 -9.6000e+01]
 [-1.4000e+01 -5.0000e+01  2.2830e+03 -4.0000e+00  1.0855e+03 -4.4500e+01
  -4.1500e+01 -1.5400e+02 -1.5250e+02 -2.3500e+01]
 [ 0.0000e+00 -8.3500e+01 -4.0000e+00  2.3000e+03 -3.2000e+01 -3.0000e+01
  -2.5000e+01 -9.8500e+01 -9.9000e+01 -2.8000e+01]
 [-2.1000e+01 -4.1500e+01  1.0855e+03 -3.2000e+01  2.3140e+03 -6.8500e+01
  -6.6000e+01 -1.9550e+02 -1.9200e+02 -7.1000e+01]
 [-3.6500e+01 -1.1050e+02 -4.4500e+01 -3.0000e+01 -6.8500e+01  1.0910e+03
   1.0750e+03 -2.2300e+02 -2.2900e+02  6.0300e+02]
 [-3.8500e+01 -1.1750e+02 -4.1500e+01 -2.5000e+01 -6.6000e+01  1.0750e+03
   1.0900e+03 -2.2300e+02 -2.2850e+02  6.1300e+02]
 [-9.3500e+01 -9.1000e+01 -1.5400e+02 -9.8500e+01 -1.9550e+02 -2.2300e+02
  -2.2300e+02  4.9530

In [11]:
def armar_arbol_guia(matriz, etiquetas, tipo_score):
      paso = 1
      while len(etiquetas) > 1:
          imprimir_matriz(matriz, etiquetas)
          if tipo_score in ['bitscore', 'pident']:
              (i, j), max_val = encontrar_maximo(matriz)
              print(f"\nPaso {paso}: Agrupando '{etiquetas[i]}' y '{etiquetas[j]}' con {tipo_score} maximo {max_val}")

          if tipo_score == 'evalue':
              (i, j), min_val = encontrar_minimo(matriz)
              print(f"\nPaso {paso}: Agrupando '{etiquetas[i]}' y '{etiquetas[j]}' con evalue mínimo {min_val}")

          matriz, etiquetas = actualizar_matriz(matriz, etiquetas, i, j)
          paso += 1

      print("\nÁrbol final:", etiquetas[0])
      return etiquetas[0]


In [12]:

evalue_matrix, bitscore_matrix, pident_matrix = obtener_matriz_blast_scores(lista_quinasas, blosum60_df)

labels = [ f"Prot {i}" for i in range(len(lista_quinasas))]
arbol_bitscore = armar_arbol_guia(bitscore_matrix, labels, 'bitscore')
labels = [ f"Prot {i}" for i in range(len(lista_quinasas))]
arbol_pident = armar_arbol_guia(pident_matrix, labels, 'pident')
labels = [ f"Prot {i}" for i in range(len(lista_quinasas))]
arbol_evalue = armar_arbol_guia(evalue_matrix, labels, 'evalue')

tree_bitscore = Phylo.read(StringIO(arbol_bitscore), "newick")
tree_pident = Phylo.read(StringIO(arbol_pident), "newick")
tree_evalue = Phylo.read(StringIO(arbol_evalue), "newick")






Matriz actual:


,Prot 0,Prot 1,Prot 2,Prot 3,Prot 4,Prot 5,Prot 6,Prot 7,Prot 8,Prot 9
Prot 0,915.0,78.2,35.4,17.3,31.2,18.1,18.1,25.8,25.8,19.6
Prot 1,77.8,1385.0,23.5,18.9,22.7,23.9,24.3,18.5,18.5,21.6
Prot 2,35.4,23.5,893.0,15.8,434.0,14.2,16.2,20.0,20.0,20.4
Prot 3,17.3,18.9,15.8,891.0,19.2,13.5,16.9,18.5,18.5,18.9
Prot 4,31.2,22.7,434.0,19.2,915.0,16.9,16.9,20.4,20.4,19.2
Prot 5,18.1,23.9,14.6,13.9,16.9,395.0,391.0,14.2,15.0,239.0
Prot 6,18.1,24.6,16.5,17.3,16.5,391.0,397.0,0.0,15.0,243.0
Prot 7,25.8,18.9,19.6,21.9,20.4,15.0,18.9,1827.0,1776.0,19.2
Prot 8,25.8,18.9,19.6,18.9,20.8,16.2,18.5,1776.0,1824.0,19.2
Prot 9,19.6,21.6,20.4,18.9,19.2,225.0,229.0,16.2,15.8,534.0



Paso 1: Agrupando 'Prot 7' y 'Prot 8' con bitscore maximo 1776.0

Matriz actual:


,Prot 0,Prot 1,Prot 2,Prot 3,Prot 4,Prot 5,Prot 6,Prot 9,"(Prot 7,Prot 8)"
Prot 0,915.0,78.2,35.4,17.3,31.2,18.1,18.1,19.6,25.8
Prot 1,77.8,1385.0,23.5,18.9,22.7,23.9,24.3,21.6,18.9
Prot 2,35.4,23.5,893.0,15.8,434.0,14.2,16.2,20.4,19.6
Prot 3,17.3,18.9,15.8,891.0,19.2,13.5,16.9,18.9,20.4
Prot 4,31.2,22.7,434.0,19.2,915.0,16.9,16.9,19.2,20.6
Prot 5,18.1,23.9,14.6,13.9,16.9,395.0,391.0,239.0,15.6
Prot 6,18.1,24.6,16.5,17.3,16.5,391.0,397.0,243.0,18.7
Prot 9,19.6,21.6,20.4,18.9,19.2,225.0,229.0,534.0,19.2
"(Prot 7,Prot 8)",25.8,18.9,19.6,20.4,20.6,15.6,18.7,19.2,0.0



Paso 2: Agrupando 'Prot 2' y 'Prot 4' con bitscore maximo 434.0

Matriz actual:


...


Paso 7: Agrupando '(Prot 7,Prot 8)' y '((Prot 2,Prot 4),(Prot 0,Prot 1))' con evalue mínimo 0.415

Matriz actual:


,Prot 3,"(Prot 9,(Prot 5,Prot 6))","((Prot 7,Prot 8),((Prot 2,Prot 4),(Prot 0,Prot 1)))"
Prot 3,0.00000,1.655000,0.865750
"(Prot 9,(Prot 5,Prot 6))",1.65500,0.000000,0.737188
"((Prot 7,Prot 8),((Prot 2,Prot 4),(Prot 0,Prot 1)))",0.86575,0.737188,0.000000



Paso 8: Agrupando '(Prot 9,(Prot 5,Prot 6))' y '((Prot 7,Prot 8),((Prot 2,Prot 4),(Prot 0,Prot 1)))' con evalue mínimo 0.7371875000000001

Matriz actual:


,Prot 3,"((Prot 9,(Prot 5,Prot 6)),((Prot 7,Prot 8),((Prot 2,Prot 4),(Prot 0,Prot 1))))"
Prot 3,0.000000,1.260375
"((Prot 9,(Prot 5,Prot 6)),((Prot 7,Prot 8),((Prot 2,Prot 4),(Prot 0,Prot 1))))",1.260375,0.000000



Paso 9: Agrupando 'Prot 3' y '((Prot 9,(Prot 5,Prot 6)),((Prot 7,Prot 8),((Prot 2,Prot 4),(Prot 0,Prot 1))))' con evalue mínimo 1.260375

Árbol final: (Prot 3,((Prot 9,(Prot 5,Prot 6)),((Prot 7,Prot 8),((Prot 2,Prot 4),(Prot 0,Prot 1)))))


In [13]:
#Muscle
tree = Phylo.read("quinasa_philo.phylotree", "newick")

tree_no_lengths = copy.deepcopy(tree)
for clade in tree_no_lengths.find_clades():
    clade.branch_length = None

Phylo.draw(tree_no_lengths)





FileNotFoundError: [Errno 2] No such file or directory: 'quinasa_philo.phylotree'

In [ ]:
#CLUSTAL
import copy
tree = Phylo.read("quinasas_completo.tree", "newick")

tree_no_lengths = copy.deepcopy(tree)
for clade in tree_no_lengths.find_clades():
    clade.branch_length = None

Phylo.draw(tree_no_lengths)


#Phylo.draw(tree_bitscore)
Phylo.draw(tree_pident)
#Phylo.draw(tree_evalue)


In [ ]:
!sudo apt-get install mafft


Con una libreria de clustering gerarquico comparamos nuestro algoritmo para ver que tan bueno es

In [ ]:
!mafft --treeout quinasas.fasta > alineamiento.aln


In [ ]:
#Muscle
tree = Phylo.read("quinasas.fasta.tree", "newick")

tree_no_lengths = copy.deepcopy(tree)
for clade in tree_no_lengths.find_clades():
    clade.branch_length = None

Phylo.draw(tree_no_lengths)








In [ ]:
!apt-get install clustalw

In [ ]:
from Bio.Align.Applications import ClustalwCommandline

cline = ClustalwCommandline(
    "clustalw",
    infile=archivo_quinasas,
    pwgapopen=10,
    pwgapext=0.5,
    matrix= 'BLOSUM' ,
)

stdout, stderr = cline()
print(stdout)
print(cline)


In [ ]:
from Bio import Phylo

tree = Phylo.read("quinasas.dnd", "newick")
Phylo.draw(tree)



In [ ]:
!clustalw -infile=quinasas.fasta -tree -outputtree=dist

with open("quinasas.dst") as f:
    content = f.read()

print(content)

lines = content.strip().split("\n")
n = int(lines[0])  # número de secuencias
names = []
matrix = []

for line in lines[1:]:
    parts = line.split()
    names.append(parts[0])
    matrix.append([float(x) for x in parts[1:]])

dist_matrix = np.array(matrix)
print("Secuencias:", names)
print("Matriz de distancias:\n", dist_matrix)



In [ ]:
import numpy as np

# Matriz de distancias 4x4
matriz = np.array([
    [0.0, 5.0, 9.0, 9.0],
    [5.0, 0.0, 10.0, 10.0],
    [9.0, 10.0, 0.0, 8.0],
    [9.0, 10.0, 8.0, 0.0]
])

# Etiquetas correspondientes
etiquetas = ['A', 'B', 'C', 'D']

print("Reduccion de la matriz:")

# === PROCESO PRINCIPAL ===
paso = 1
while len(etiquetas) > 1:
    imprimir_matriz(matriz, etiquetas)
    (i, j), max_val = encontrar_maximo(matriz)
    print(f"\nPaso {paso}: Agrupando '{etiquetas[i]}' y '{etiquetas[j]}' con distancia maxima {max_val:.2f}")
    matriz, etiquetas = actualizar_matriz(matriz, etiquetas, i, j)
    paso += 1

print("\nÁrbol final:", etiquetas[0])


In [ ]:
labels = [ f"Prot {i}" for i in range(len(lista_quinasas))]
arbol_pident = armar_arbol_guia(pident_matrix, labels, 'pident')
tree_pident = Phylo.read(StringIO(arbol_pident), "newick")
Phylo.draw(tree_pident)

In [ ]:
tree = Phylo.read("quinasas_completo.tree", "newick")
tree_no_lengths = copy.deepcopy(tree)
for clade in tree_no_lengths.find_clades():
    clade.branch_length = None

Phylo.draw(tree_no_lengths)
